In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number, rank, dense_rank, sum

spark = SparkSession.builder.appName("WindowExample").getOrCreate()


data = [("Ravi", "IT", 60000),
        ("Yaswanth", "IT", 10000),
        ("Anita", "HR", 45000),
        ("Kiran", "HR", 30000)]

df = spark.createDataFrame(data, ["name", "dept", "salary"])

# Define window partitioned by dept, ordered by salary descending
windowSpec = Window.partitionBy("dept").orderBy(col("salary").desc())

df_ranked = df.withColumn("row_number", row_number().over(windowSpec)) \
              .withColumn("rank", rank().over(windowSpec)) \
              .withColumn("dense_rank", dense_rank().over(windowSpec))

df_ranked.show()



+--------+----+------+----------+----+----------+
|    name|dept|salary|row_number|rank|dense_rank|
+--------+----+------+----------+----+----------+
|   Anita|  HR| 45000|         1|   1|         1|
|   Kiran|  HR| 30000|         2|   2|         2|
|    Ravi|  IT| 60000|         1|   1|         1|
|Yaswanth|  IT| 10000|         2|   2|         2|
+--------+----+------+----------+----+----------+



In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number

windowSpec = Window.partitionBy("dept").orderBy(col("salary").desc())

df_rownum = df.withColumn("row_number", row_number().over(windowSpec))
df_rownum.show()


+--------+----+------+----------+
|    name|dept|salary|row_number|
+--------+----+------+----------+
|   Anita|  HR| 45000|         1|
|   Kiran|  HR| 30000|         2|
|    Ravi|  IT| 60000|         1|
|Yaswanth|  IT| 10000|         2|
+--------+----+------+----------+



In [0]:
from pyspark.sql.functions import rank

df_rank = df.withColumn("rank", rank().over(windowSpec))
df_rank.show()



+--------+----+------+----+
|    name|dept|salary|rank|
+--------+----+------+----+
|   Anita|  HR| 45000|   1|
|   Kiran|  HR| 30000|   2|
|    Ravi|  IT| 60000|   1|
|Yaswanth|  IT| 10000|   2|
+--------+----+------+----+



In [0]:
from pyspark.sql.functions import dense_rank

df_dense = df.withColumn("dense_rank", dense_rank().over(windowSpec))
df_dense.show()


+--------+----+------+----------+
|    name|dept|salary|dense_rank|
+--------+----+------+----------+
|   Anita|  HR| 45000|         1|
|   Kiran|  HR| 30000|         2|
|    Ravi|  IT| 60000|         1|
|Yaswanth|  IT| 10000|         2|
+--------+----+------+----------+



In [0]:
##lag()

from pyspark.sql.functions import lag
from pyspark.sql import Window

windowSpec = Window.partitionBy("dept").orderBy("salary")

df_lag = df.withColumn("prev_salary", lag("salary", 1).over(windowSpec))
df_lag.show()


#lead()
from pyspark.sql.functions import lead

df_lead = df.withColumn("next_salary", lead("salary", 1).over(windowSpec))
df_lead.show()

#ntile(n)
from pyspark.sql.functions import ntile

df_ntile = df.withColumn("quartile", ntile(4).over(windowSpec))
df_ntile.show()



+--------+----+------+-----------+
|    name|dept|salary|prev_salary|
+--------+----+------+-----------+
|   Kiran|  HR| 30000|       NULL|
|   Anita|  HR| 45000|      30000|
|Yaswanth|  IT| 10000|       NULL|
|    Ravi|  IT| 60000|      10000|
+--------+----+------+-----------+

+--------+----+------+-----------+
|    name|dept|salary|next_salary|
+--------+----+------+-----------+
|   Kiran|  HR| 30000|      45000|
|   Anita|  HR| 45000|       NULL|
|Yaswanth|  IT| 10000|      60000|
|    Ravi|  IT| 60000|       NULL|
+--------+----+------+-----------+

+--------+----+------+--------+
|    name|dept|salary|quartile|
+--------+----+------+--------+
|   Kiran|  HR| 30000|       1|
|   Anita|  HR| 45000|       2|
|Yaswanth|  IT| 10000|       1|
|    Ravi|  IT| 60000|       2|
+--------+----+------+--------+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql.functions import col, lag, lead, ntile

spark = SparkSession.builder.appName("AnalyticFunctionsExample").getOrCreate()

data = [
    ("Ravi", "IT", 60000),
    ("Yaswanth", "IT", 10000),
    ("Anita", "HR", 45000),
    ("Kiran", "HR", 30000),
    ("Meena", "IT", 70000),
    ("Suresh", "HR", 25000),
    ("Divya", "Finance", 55000),
    ("Arjun", "Finance", 40000),
    ("Priya", "Finance", 65000),
    ("Nikhil", "IT", 50000)
]

columns = ["name", "dept", "salary"]
df = spark.createDataFrame(data, columns)


In [0]:
windowSpec = Window.partitionBy("dept").orderBy("salary")

df_lag = df.withColumn("prev_salary", lag("salary", 1).over(windowSpec))
df_lag.show()


df_lead = df.withColumn("next_salary", lead("salary", 1).over(windowSpec))
df_lead.show()


df_ntile = df.withColumn("quartile", ntile(4).over(windowSpec))
df_ntile.show()


+--------+-------+------+-----------+
|    name|   dept|salary|prev_salary|
+--------+-------+------+-----------+
|   Arjun|Finance| 40000|       NULL|
|   Divya|Finance| 55000|      40000|
|   Priya|Finance| 65000|      55000|
|  Suresh|     HR| 25000|       NULL|
|   Kiran|     HR| 30000|      25000|
|   Anita|     HR| 45000|      30000|
|Yaswanth|     IT| 10000|       NULL|
|  Nikhil|     IT| 50000|      10000|
|    Ravi|     IT| 60000|      50000|
|   Meena|     IT| 70000|      60000|
+--------+-------+------+-----------+

+--------+-------+------+-----------+
|    name|   dept|salary|next_salary|
+--------+-------+------+-----------+
|   Arjun|Finance| 40000|      55000|
|   Divya|Finance| 55000|      65000|
|   Priya|Finance| 65000|       NULL|
|  Suresh|     HR| 25000|      30000|
|   Kiran|     HR| 30000|      45000|
|   Anita|     HR| 45000|       NULL|
|Yaswanth|     IT| 10000|      50000|
|  Nikhil|     IT| 50000|      60000|
|    Ravi|     IT| 60000|      70000|
|   Meena| 